# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [13]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [21]:
df['revenue'] = df['qty'] * df['price']       #new revenue column added to df
print('Total Revenue:', df['revenue'].sum())
print('Total Units:', df['qty'].sum())

Total Revenue: 8520.0
Total Units: 783


The first value is the Total Revenue, which has been calculated by taking the sum of the newly added revenue column and the second value is the sum of all the order qty.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [15]:
df['revenue_percent'] = (df['revenue'] / df['revenue'].sum() * 100).round(2)    # new column, revenue_percent, added to df
revenue_by_category = (df.groupby('category')                                    #new dataframe, revenue_by_category, created
                       .agg(revenue_percent = ('revenue_percent', 'sum'),
                            revenue = ('revenue', 'sum'),
                            orders = ('qty','sum'))
                       .round(2)
                       .sort_values('revenue_percent', ascending = False)
                      )
revenue_by_category = revenue_by_category.reset_index()                         #make it so that the category isn't the indexing value
print(df.head())
print()
revenue_by_category

  vendor_id  category  qty  price  revenue  revenue_percent
0      V-10     Drink    2   24.0     48.0             0.56
1      V-18  RainGear    1   12.0     12.0             0.14
2      V-18     Drink    3    4.5     13.5             0.16
3      V-10      Food    2   12.0     24.0             0.28
4      V-18     Drink    3    7.5     22.5             0.26



,category,revenue_percent,revenue,orders
0,Food,50.31,4293.0,362
1,Merch,20.75,1771.5,158
2,Drink,18.24,1554.0,178
3,RainGear,10.55,901.5,85


First dataframe is original dataframe with revenue_percent column added. Second dataframe is a new dataframe called by revenue_by_category which includes the revenue percent for, revenue, and orders for each category.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [16]:
revenue_by_category['average_revenue'] = (revenue_by_category['revenue'] / revenue_by_category['orders']).round(2)
revenue_by_category

,category,revenue_percent,revenue,orders,average_revenue
0,Food,50.31,4293.0,362,11.86
1,Merch,20.75,1771.5,158,11.21
2,Drink,18.24,1554.0,178,8.73
3,RainGear,10.55,901.5,85,10.61


Average revenue per order added to revenue_by_category dataframe.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [17]:
print('Percent of revenue from Merch:', f"{revenue_by_category.loc[revenue_by_category['category'] == 'Merch', 'revenue_percent'].iloc[0]:.1f}%")

Percent of revenue from Merch: 20.8%


Percent of revenue from merch is reported

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [23]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

rows_before = len(df)
revenue_before = df['revenue'].sum()

joined = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)

unmatched = joined.loc[joined['vendor_name'].isna(), 'vendor_id'].unique()

rows_after = len(joined)
revenue_after = joined['revenue'].sum()

print("Unmatched vendor:", unmatched)
print("Rows Before:", rows_before, "| Rows After", rows_after)
print("Revenue Before:", revenue_before, "| Revenue After:", revenue_after)
print("Row count unchanged:", rows_before == rows_after)
print("Revenue unchanged:", revenue_before == revenue_after)

Unmatched vendor: ['V-18']
Rows Before: 400 | Rows After 400
Revenue Before: 8520.0 | Revenue After: 8520.0
Row count unchanged: True
Revenue unchanged: True


**The unmatched vendor, and what I did about it:** _..._

The unmatched vendor was V-18, I identified it and used left join to keep it in in order to keep the rows and revenue from changing.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [19]:
pivot = pd.pivot_table(
    joined,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    margins=True,
    margins_name='Total',
    fill_value=0
)

pivot

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Total,972.0,3274.5,1263.0,661.5,6171.0


Dataframe shows each vendors revenue by category and the totals of their revenue and the category total revenue

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [20]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(revenue_by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

**a)** I would tell the vendors to focus on their food sales because that is where they get the most revenue, $$3274.50 / $6171.00 or over half. Cav Merch North should aslo focus on its drink sales, because it far outcompetes with over half of drink sales ($502.5 / $972.00).

**b)** My least trustworthy answers is Q6 because V-18 was unmatched in the vendor lookup. This means its sales could not be associated with a vendor name, making some vendor comparisons less reliable.